# K08_05 – SVM Gesichtserkennung mit PCA (Student)

## Lernziele
Nach diesem Notebook können Sie:

- ein klassisches Beispiel für Gesichtserkennung mit SVM einordnen
- PCA als Vorverarbeitung für hochdimensionale Bilddaten verstehen
- eine Pipeline aus PCA und SVM anwenden
- Grid Search für `C` und `gamma` interpretieren

## Didaktischer Hinweis

Dieses Notebook basiert auf dem klassischen Datensatz  
**Labeled Faces in the Wild (LFW)**.

Wichtig:
- Das Laden der Daten kann eine Internetverbindung benötigen.
- Für Google Colab ist dieses Beispiel in der Regel gut geeignet.
- Wenn die Daten lokal nicht verfügbar sind, sollte das Notebook als **Vorbereitungs- oder Zusatzmaterial** verwendet werden.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## 1. Gelabelte Gesichter laden

In [ ]:
faces = fetch_lfw_people(min_faces_per_person=60, resize=0.4)

print("Klassen:", faces.target_names)
print("Bildmatrix:", faces.images.shape)
print("Feature-Matrix:", faces.data.shape)

In [ ]:
fig, ax = plt.subplots(3, 5, figsize=(8, 6))
for i, axi in enumerate(ax.flat):
    axi.imshow(faces.images[i], cmap='bone')
    axi.set(xticks=[], yticks=[],
            xlabel=faces.target_names[faces.target[i]])
plt.tight_layout()
plt.show()

## 2. Warum brauchen wir hier PCA?

Jedes Bild enthält sehr viele Pixelmerkmale.  
Eine direkte SVM auf Rohpixeln wäre oft unnötig teuer und schwerer zu interpretieren.

PCA hilft hier:
- Dimension reduzieren
- unkorrelierte Hauptkomponenten erzeugen
- die SVM auf einen kompakteren Merkmalsraum anzuwenden

In [ ]:
pca = PCA(n_components=120, whiten=True, svd_solver='randomized', random_state=42)
svc = SVC(kernel='rbf', class_weight='balanced')
model = make_pipeline(pca, svc)

## 3. Trainings- und Testdaten

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(
    faces.data, faces.target, random_state=42, stratify=faces.target
)

## 4. Grid Search für `C` und `gamma`

In [ ]:
param_grid = {
    'svc__C': [1, 5, 10, 50],
    'svc__gamma': [0.0001, 0.0005, 0.001, 0.005]
}

grid = GridSearchCV(model, param_grid)
grid.fit(Xtrain, ytrain)

print("Beste Parameter:", grid.best_params_)

## 5. Vorhersage mit dem besten Modell

In [ ]:
best_model = grid.best_estimator_
yfit = best_model.predict(Xtest)

print(classification_report(ytest, yfit, target_names=faces.target_names))

## 6. Konfusionsmatrix

In [ ]:
cm = confusion_matrix(ytest, yfit)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=faces.target_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45)
plt.show()

## 7. Längere Übungsaufgabe

1. Erklären Sie, warum PCA hier ein sinnvoller Vorverarbeitungsschritt ist.

2. Interpretieren Sie die Grid Search:
   - Warum suchen wir sowohl `C` als auch `gamma`?
   - Was bedeutet es, wenn die besten Werte am Rand des Gitters liegen?

3. Analysieren Sie die Konfusionsmatrix:
   - Welche Personen werden besonders gut erkannt?
   - Welche Verwechslungen treten auf?

4. Schreiben Sie eine kurze Reflexion:
   - Warum waren PCA + SVM lange ein starkes Standardverfahren für Gesichtserkennung?
   - Welche Grenzen hat dieses Verfahren im Vergleich zu modernen Deep-Learning-Ansätzen?

## 8. Fazit

- Gesichtserkennung mit SVM ist ein klassisches Beispiel für hochdimensionale Klassifikation.
- PCA reduziert die Bilddaten auf einen kompakteren Merkmalsraum.
- Grid Search hilft, `C` und `gamma` systematisch abzustimmen.
- Das Beispiel zeigt sehr gut die klassische Stärke von SVMs vor dem Deep-Learning-Zeitalter.